<a href="https://colab.research.google.com/github/cermegno/llm-evals/blob/main/01-basic-SQUAD-evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic QA evaluation
We will use SQuAD (Stanford Question Answering Dataset) metrics: F1, Precision, Recall and Exact Match

Intuition:
- **Precision**: "*What fraction of what I SAID was right?*". Precision penalizes extra, irrelevant stuff
- **Recall**: "*What fraction of what I SHOULD have said did I actually say?"*. Recall penalizes missing pieces of the true answer

**F1** is the harmonic mean of precision and recall.
- F1 is high only when both precision and recall are reasonably high.
- If either precision or recall is low, F1 is dragged down.

In [28]:
!pip install langchain langchain-nvidia-ai-endpoints langchain-core pandas --quiet

In [29]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
import json
import pandas as pd

## Nvidia setup

In [63]:
# Initialize the LLM
from google.colab import userdata
apikey = userdata.get('apikey')
os.environ["NVIDIA_API_KEY"] = apikey
golden_llm = ChatNVIDIA(model="openai/gpt-oss-120b")
llm_to_test = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Respond concisely, as simple as possible, in plain text, no markdown."),
    ("user", "{input}")
])

golden_chain = prompt | golden_llm

prediction_chain = prompt | llm_to_test

## Create Dataset

In [30]:
questions = [
    "What is the capital of Australia",
    "How much is 2 * 24",
    "What are the ingredients of basic bread",
    "Translate 'I am hungry' to spanish",
    "Name the benefits of eating healthy"
]

dataset = []
for q in questions:
  response = golden_chain.invoke({"input": q})
  dataset.append({"question": q, "golden_answer": response.content})

In [31]:
print(json.dumps(dataset, indent=2))

[
  {
    "question": "What is the capital of Australia",
    "golden_answer": "Canberra."
  },
  {
    "question": "How much is 2 * 24",
    "golden_answer": "48"
  },
  {
    "question": "What are the ingredients of basic bread",
    "golden_answer": "A basic loaf of bread typically uses:\n\n- Flour (usually wheat flour)  \n- Water  \n- Yeast (active dry or instant)  \n- Salt  "
  },
  {
    "question": "Translate 'I am hungry' to spanish",
    "golden_answer": "Tengo hambre."
  },
  {
    "question": "Name the benefits of eating healthy",
    "golden_answer": "- Improved energy levels  \n- Better weight management  \n- Lower risk of chronic diseases (heart disease, diabetes, certain cancers)  \n- Stronger immune system  \n- Improved mood and mental health  \n- Enhanced digestion and gut health  \n- Stronger bones and teeth  \n- Healthier skin, hair, and nails  \n- Greater longevity and quality of life"
  }
]


## Define evaluation metrics
Exact match, Precision, Recall and F1


In [69]:
import re

def normalize(text: str) -> str:
  """break into words and cleanup"""
  text = text.lower().strip()
  text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
  text = re.sub(r"\s+", " ", text)      # collapse spaces
  return text
#print(normalize("I am  hungry, TIRED! and; you?"))

def exact_match(pred: str, gold: str) -> bool:
  return normalize(pred) == normalize(gold)

#print(exact_match("I am  HUNGRY!!!, and you", "I am hungry: and you;?"))
#print(exact_match("canberra", dataset[0]["golden_answer"]))

def f1_score(pred: str, gold: str):

  pred_norm = normalize(pred)
  gold_norm = normalize(gold)

  p_tokens = pred_norm.split()
  g_tokens = gold_norm.split()

  if verbose:
    print("Prediction tokens:", p_tokens)
    print("Gold tokens:", g_tokens)

  # Find which token types appear in both
  p_set = set(p_tokens)
  g_set = set(g_tokens)
  common = p_set & g_set

  if verbose: print("Common tokens:", common)

  # Count how many overlapping tokens (respecting duplicates)
  num_common = 0
  for t in common:
      count_pred = p_tokens.count(t)
      count_gold = g_tokens.count(t)
      overlap_for_t = min(count_pred, count_gold)

      if verbose:
        print(f"Token '{t}': pred has {count_pred}, gold has {count_gold}, overlap = {overlap_for_t}")

      num_common += overlap_for_t

  if verbose: print("Total overlapping tokens:", num_common)

  precision = num_common / len(p_tokens) if len(p_tokens) > 0 else 0.0
  recall = num_common / len(g_tokens) if len(g_tokens) > 0 else 0.0

  if precision == 0 and recall == 0:
      f1 = 0.0
  else:
      f1 = 2 * precision * recall / (precision + recall)

  return round(precision, 2), round(recall, 2), round(f1, 2)

verbose = 1
result = f1_score("Yo tengo hambre, Mucha HAMBRE!!", dataset[3]["golden_answer"])
for k, v in zip(["precision", "recall", "f1"], result):
  print(f" - {k.upper()}: {v}")


Prediction tokens: ['yo', 'tengo', 'hambre', 'mucha', 'hambre']
Gold tokens: ['tengo', 'hambre']
Common tokens: {'hambre', 'tengo'}
Token 'hambre': pred has 2, gold has 1, overlap = 1
Token 'tengo': pred has 1, gold has 1, overlap = 1
Total overlapping tokens: 2
 - PRECISION: 0.4
 - RECALL: 1.0
 - F1: 0.57


## Test the model

In [72]:
verbose = 0
# Add predictions, and metrics to dataset: precision, recall, f1, exact match
for q in dataset:
  response = prediction_chain.invoke({"input": q})
  q["prediction"] = response.content
  q["precision"], q["recall"], q["f1"] = f1_score(response.content, q["golden_answer"])
  q["exact_match"] = exact_match(response.content, q["golden_answer"])


The conciseness instructions in the prompt are not doing the trick for the little Llama 3.1 model, so it is getting penalized in *precision*, which in turn impacts the F1 score

In [73]:
df = pd.DataFrame(dataset)
display(df)

,question,golden_answer,prediction,precision,recall,f1,exact_match
0,What is the capital of Australia,Canberra.,The capital of Australia is Canberra.,0.17,1.00,0.29,False
1,How much is 2 * 24,48,The correct calculation is 2 * 24 = 48.,0.14,1.00,0.25,False
2,What are the ingredients of basic bread,A basic loaf of bread typically uses:\n\n- Flo...,"The basic ingredients of bread are flour, wate...",0.64,0.39,0.48,False
3,Translate 'I am hungry' to spanish,Tengo hambre.,"The translation of ""I am hungry"" to Spanish is...",0.18,1.00,0.31,False
4,Name the benefits of eating healthy,- Improved energy levels \n- Better weight ma...,The benefits of eating healthy include improve...,0.85,0.89,0.87,False


In [78]:
avg_metrics = df[['precision', 'recall', 'f1', 'exact_match']].mean()
avg_metrics.name = "Scores"
display(avg_metrics)

,Scores
precision,0.396
recall,0.856
f1,0.440
exact_match,0.000
